# Synthetic control: Proposition 99 làm California hút thuốc ít đi bao nhiêu?

Năm 1988 California thông qua Proposition 99 (thuế thêm 25 cent mỗi bao, chiến dịch chống hút
thuốc), có hiệu lực từ 1989. Chỉ có **một** bang được can thiệp, và bài Basic 02 đã cho thấy
California giảm nhanh hơn các bang khác từ trước 1989, nên DiD với "trung bình 38 bang" không đáng tin.

Synthetic control (Abadie, Diamond, Hainmueller 2010) dựng một "California tổng hợp": tổ hợp có
trọng số của các bang khác, trọng số không âm và cộng lại bằng 1, chọn sao cho bám sát California
**trước** 1989. Sau 1989, khoảng cách giữa California thật và California tổng hợp là tác động.

Dữ liệu: `prop99/smoking_data.csv`, cột `state`, `year`, `cigsale` (số bao bán trên đầu người).

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm.
- Cell gắn tag `explore` (dòng đầu `# explore`) đọc dữ liệu thật, vẽ, gọi hàm. Test bỏ qua chúng.
- Chấm: `uv run pytest tracks/data-science/modules/06-causal-inference/exercises/intermediate/02-synthetic-control-california -q`

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import nnls, minimize

In [ ]:
# explore: đọc dữ liệu thật (tải bằng: uv run python data/download_dataset.py prop99)
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
raw = pd.read_csv(ROOT / "data/raw/prop99/smoking_data.csv")
raw["year"] = raw["year"].astype(int)
PRE = list(range(1970, 1989))
POST = list(range(1989, 2001))
raw.head()

## 1. Panel rộng

Mỗi dòng là một năm, mỗi cột là một bang.

In [ ]:
def to_panel(df: pd.DataFrame, unit: str = "state", time: str = "year", outcome: str = "cigsale") -> pd.DataFrame:
    """Index là time (tăng dần), cột là unit, giá trị là outcome."""
    return df.pivot(index=time, columns=unit, values=outcome).sort_index()

In [ ]:
# explore: California so với trung bình các bang khác
import matplotlib.pyplot as plt

panel = to_panel(raw)
ax = panel["California"].plot(label="California")
panel.drop(columns="California").mean(axis=1).plot(ax=ax, label="Trung bình 38 bang")
ax.axvline(1988.5, ls="--", c="gray"); ax.set_ylabel("cigsale"); ax.legend(); plt.show()

## 2. Trọng số synthetic control

Tìm $$w \ge 0$$, $$\sum_j w_j = 1$$ tối thiểu hoá
$$\sum_{t \in \text{pre}} \left(y_{1t} - \sum_j w_j y_{jt}\right)^2$$.

Bản gốc của Abadie còn khớp cả covariate (giá thuốc, thu nhập, tỉ lệ người trẻ, bia) với một ma
trận trọng số V. Bài này chỉ khớp chính outcome trước can thiệp: đơn giản hơn, thường cho kết quả gần.

In [ ]:
def synth_weights(y_pre: pd.Series, donors_pre: pd.DataFrame) -> pd.Series:
    """Trọng số không âm, tổng bằng 1, index là tên donor (cột của donors_pre)."""
    A = donors_pre.to_numpy(dtype=float)
    b = y_pre.to_numpy(dtype=float)
    big = 1e4 * max(1.0, np.abs(A).max())                 # dòng phạt lớn ép tổng trọng số về 1
    w, _ = nnls(np.vstack([A, big * np.ones(A.shape[1])]), np.append(b, big), maxiter=50 * A.shape[1])
    return pd.Series(w / w.sum(), index=donors_pre.columns)


def synth_gap(panel: pd.DataFrame, treated: str, pre_years: list[int]) -> pd.Series:
    """y thật − y tổng hợp của `treated` cho mọi năm. Donor là mọi cột khác trong panel."""
    donors = panel.drop(columns=treated)
    w = synth_weights(panel.loc[pre_years, treated], donors.loc[pre_years])
    return panel[treated] - donors @ w


def rmspe(gap: pd.Series, years: list[int]) -> float:
    """Root mean squared prediction error của gap trên các năm cho trước."""
    return float(np.sqrt(np.mean(gap.loc[years].to_numpy(dtype=float) ** 2)))

In [ ]:
# explore: trọng số, độ khớp trước 1989, khoảng cách sau
w = synth_weights(panel.loc[PRE, "California"], panel.drop(columns="California").loc[PRE])
print(w[w > 0.001].sort_values(ascending=False).round(3))
gap = synth_gap(panel, "California", PRE)
print("RMSPE trước:", round(rmspe(gap, PRE), 2), "| sau:", round(rmspe(gap, POST), 2))
print("Gap 1989, 1995, 2000:", gap.loc[[1989, 1995, 2000]].round(1).to_dict())
print("Gap trung bình 1989–2000:", round(gap.loc[POST].mean(), 1))
synthetic = panel["California"] - gap
ax = panel["California"].plot(label="California")
synthetic.plot(ax=ax, ls="--", label="California tổng hợp")
ax.axvline(1988.5, ls="--", c="gray"); ax.legend(); plt.show()

## 3. Suy luận bằng placebo

Chỉ có một đơn vị treated, nên không có standard error theo nghĩa thông thường. Thay vào đó: giả vờ
**từng bang khác** là bang được can thiệp (donor là mọi bang còn lại **trừ California**), tính lại
synthetic control, và so sánh. Thống kê dùng tỉ số RMSPE sau / RMSPE trước: bang nào khớp kém từ
đầu thì gap sau lớn cũng không có gì lạ, tỉ số chuẩn hoá điều đó.

p-value hoán vị = (thứ hạng của California theo tỉ số, giảm dần) / (số bang).

In [ ]:
def placebo_ratios(panel: pd.DataFrame, treated: str, pre_years: list[int], post_years: list[int]) -> pd.Series:
    """Tỉ số rmspe(post) / rmspe(pre) cho mọi đơn vị, sắp giảm dần."""
    ratios = {}
    for unit in panel.columns:
        sub = panel if unit == treated else panel.drop(columns=treated)
        gap = synth_gap(sub, unit, pre_years)
        ratios[unit] = rmspe(gap, post_years) / rmspe(gap, pre_years)
    return pd.Series(ratios).sort_values(ascending=False)

In [ ]:
# explore: California đứng thứ mấy?
ratios = placebo_ratios(panel, "California", PRE, POST)
rank = list(ratios.index).index("California") + 1
print(ratios.head(5).round(2))
print(f"California hạng {rank}/{len(ratios)} → p = {rank / len(ratios):.3f}")

In [ ]:
# explore: kiểm tra độ bền. (1) Bỏ từng donor có trọng số lớn (leave-one-out). (2) Placebo theo thời gian: giả vờ can thiệp năm 1980
for drop in w[w > 0.05].index:
    g = synth_gap(panel.drop(columns=drop), "California", PRE)
    print(f"bỏ {drop:<14} gap 2000 = {g.loc[2000]:6.1f}")
fake_pre, fake_post = list(range(1970, 1980)), list(range(1980, 1989))
g = synth_gap(panel.loc[1970:1988], "California", fake_pre)
print("Placebo thời gian 1980: gap trung bình 1980–1988 =", round(g.loc[fake_post].mean(), 1))

## 4. Kết luận (bài mẫu)

California tổng hợp gồm Utah (0.39), Montana (0.23), Nevada (0.20), Connecticut (0.11), New
Hampshire và Colorado; nó bám California rất sát trước 1989 (RMSPE 1.66 bao/người). Sau 1989 khoảng
cách mở dần: −8.4 bao năm 1989, −22.9 năm 1995, −26.6 năm 2000, trung bình −19.5 bao/người/năm trong
giai đoạn 1989–2000. Con số này nhỏ hơn DiD với trung bình 38 bang (−27.3), vì DiD tính cả xu hướng
giảm sẵn có của California vào tác động.

Placebo: tỉ số RMSPE sau/trước của California là 12.4, đứng thứ 3/39 (p = 0.077), sau Missouri và
Virginia, hai bang khớp trước can thiệp rất sát (RMSPE trước dưới 1) nên tỉ số bị thổi lên. Kết quả
gốc của Abadie và cộng sự, có khớp thêm covariate, xếp California thứ nhất (p = 1/39). Kết luận
trung thực: bằng chứng mạnh về hướng và độ lớn, nhưng với 39 đơn vị thì p-value hoán vị không thể
nhỏ hơn 0.026, và lựa chọn đặc tả (covariate, donor pool) làm thứ hạng thay đổi.